# Notebook - Camada Silver

Criação de tabelas de dimensão de clientes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_customers")

#Mapeamento de colunas
df_silver = df_bronze.select(
  F.col("customer_id").cast("string").alias("id_consumidor"),
  F.col("customer_unique_id").cast("string").alias("id_unica_consumidor"),
  F.col("customer_zip_code_prefix").cast("string").alias("prefixo_cep_consumidor"),
  F.col("customer_city").alias("cidade_consumidor"),
  F.col("customer_state").alias("estado_consumidor"),
  F.col("timestamp_ingestion").alias("carimbo_data_ingestao_bronze")
)

#Limpeza e organização de dados
df_silver = (
  df_silver
  #Garantir que as cidades e estados inseridas com primeira letra maiúscula
  .withColumn("cidade_consumidor", F.upper(F.trim(F.col("cidade_consumidor"))))
  .withColumn("estado_consumidor", F.upper(F.trim(F.col("estado_consumidor"))))
  #Garantir que o prefixo do CEP tenha sempre 5 caracteres i.e. não perca o zero a esquerda
  .withColumn("prefixo_cep_consumidor", F.lpad(F.col("prefixo_cep_consumidor"),5, "0")) 
  .filter(F.col("id_consumidor").isNotNull()) #Filtrando ids vazios

)

window_dedup = Window.partitionBy("id_consumidor").orderBy(F.col("carimbo_data_ingestao_bronze").desc())

df_silver = (
    df_silver
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("id_consumidor").isNull()).count()
print(f"Total de registros nulos: {nulls}")
duplicates = total - df_silver.select("id_consumidor").distinct().count()
print(f"Total de registros duplicados: {duplicates}")

assert nulls == 0, "Existem registros nulos na tabela"
assert duplicates == 0, "Existem registros duplicados na tabela"

TABLE_NAME = "ecommerce_lakehouse.silver.dim_consumidores"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)



Total de registros: 99441
Total de registros nulos: 0
Total de registros duplicados: 0


Criação de tabelas fato de pedidos para silver

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_orders")

df_silver = df_bronze.select(
    F.col("order_id").cast("string").alias("id_pedido"),
    F.col("customer_id").cast("string").alias("id_consumidor"),
    F.col("order_status").alias("status_pedido"),
    F.col("order_purchase_timestamp").cast("timestamp").alias("data_compra"),
    F.col("order_approved_at").cast("timestamp").alias("data_aprovacao"),
    F.col("order_delivered_carrier_date").cast("timestamp").alias("data_entrega_transportadora"),
    F.col("order_delivered_customer_date").cast("timestamp").alias("data_entrega_consumidor"),
    F.col("order_estimated_delivery_date").cast("timestamp").alias("data_entrega_estimada"),
    F.col("timestamp_ingestion").alias("carimbo_data_ingestao_bronze")
)

df_silver = df_silver.withColumn(
    "status_pedido",
    F.when(F.col("status_pedido") == "delivered", "Entregue")
     .when(F.col("status_pedido") == "canceled", "Cancelado")
     .when(F.col("status_pedido") == "shipped", "Enviado")
     .when(F.col("status_pedido") == "processing", "Processando")
     .when(F.col("status_pedido") == "invoiced", "Faturado")
     .when(F.col("status_pedido") == "unavailable", "Indisponível")
     .when(F.col("status_pedido") == "created", "Criado")
     .when(F.col("status_pedido") == "approved", "Aprovado")
     .otherwise(F.lit("Desconhecido"))
).filter(F.col("id_pedido").isNotNull() & F.col("id_consumidor").isNotNull()) 

df_silver = (
    df_silver
    .withColumn(
        "tempo_entrega_dias", 
        F.datediff(F.col("data_entrega_consumidor"), F.col("data_compra"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_compra"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.datediff(F.col("data_entrega_consumidor"), F.col("data_entrega_estimada"))
    )
    .withColumn(
        "entrega_no_prazo",
        F.when(F.col("diferenca_entrega_dias") <= 0, "Sim")
        .otherwise("Não")
    )

)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("id_pedido").isNull()).count()
print(f"Total de registros nulos: {nulls}")
duplicates = total - df_silver.select("id_pedido").distinct().count()
print(f"Total de registros duplicados: {duplicates}")

assert nulls == 0, "Existem registros nulos na tabela"

TABLE_NAME = "ecommerce_lakehouse.silver.fat_pedidos"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)

Total de registros: 99441
Total de registros nulos: 0
Total de registros duplicados: 0


Criação de tabelas fato de items por pedido

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_order_items")

df_silver = df_bronze.select(
    F.col("order_id").cast("string").alias("id_pedido"),
    F.col("product_id").cast("string").alias("id_produto"),
    F.col("seller_id").cast("string").alias("id_vendedor"),
    F.col("order_item_id").cast("int").alias("id_item"),
    F.col("shipping_limit_date").cast("timestamp").alias("data_limite_entrega"),
    F.col("price").cast("double").alias("preco_BRL"),
    F.col("freight_value").cast("double").alias("preco_frete"),
    F.col("timestamp_ingestion").alias("carimbo_data_ingestao_bronze")
)

df_silver.filter(F.col("id_pedido").isNotNull() & F.col("id_produto").isNotNull())

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("id_pedido").isNull()).count()
print(f"Total de registros nulos: {nulls}")

assert nulls == 0, "Existem registros nulos na tabela"

TABLE_NAME = "ecommerce_lakehouse.silver.fat_pedidos_itens"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)

Total de registros: 112650
Total de registros nulos: 0


Criação de tabela de fato de pagamentos por pedido

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_order_payments")

df_silver = df_bronze.select(
    F.col("order_id").cast("string").alias("id_pedido"),
    F.col("payment_sequential").cast("int").alias("sequencia_pagamento"),
    F.col("payment_type").cast("string").alias("tipo_pagamento"),
    F.col("payment_installments").cast("int").alias("parcelas"),
    F.col("payment_value").cast("double").alias("valor"),
    F.col("timestamp_ingestion").alias("carimbo_data_ingestao_bronze")
)

df_silver = df_silver.withColumn(
    "tipo_pagamento",
    F.when(F.col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
     .when(F.col("tipo_pagamento") == "boleto", "Boleto")
     .when(F.col("tipo_pagamento") == "voucher", "Voucher")
     .when(F.col("tipo_pagamento") == "debit_card", "Cartão de Débito")
     .otherwise(F.lit("Não Definido"))
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("id_pedido").isNull()).count()
print(f"Total de registros nulos: {nulls}")

assert nulls == 0, "Existem registros nulos na tabela"

TABLE_NAME = "ecommerce_lakehouse.silver.fat_pagamentos_pedidos"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)

Total de registros: 103886
Total de registros nulos: 0


Criação de tabela fato de avaliações por pedido

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_order_reviews")

df_silver = df_bronze.select(
    F.col("review_id").cast("string").alias("id_avaliacao"),
    F.col("order_id").cast("string").alias("id_pedido"),
    F.col("review_score").try_cast("int").alias("nota_avaliacao"),
    F.col("review_comment_title").cast("string").alias("titulo_comentario_avaliacao"),
    F.col("review_comment_message").cast("string").alias("comentario_avaliacao"),
    F.try_to_timestamp(F.col("review_creation_date"), F.lit("yyyy-MM-dd HH:mm:ss"))
    .alias("data_criacao_avaliacao"),
    F.try_to_timestamp(F.col("review_answer_timestamp"), F.lit("yyyy-MM-dd HH:mm:ss"))
    .alias("data_resposta_avaliacao"),
    F.col("timestamp_ingestion").alias("carimbo_data_ingestao_bronze")
)

data_atual = F.current_timestamp()

#Limpeza da tabela
df_silver = (
    df_silver
    #filtrando as datas falhas do try_to_timestamp
    .filter(F.col("data_criacao_avaliacao").isNotNull() | F.col("data_resposta_avaliacao").isNotNull())
    #id_pedido não pode ser nulo
    .filter(F.trim(F.col("id_pedido")) != "")
    .filter(F.col("id_pedido").isNotNull())
    #id_avaliacao não pode ser nulo
    .filter(F.trim(F.col("id_avaliacao")) != "")
    .filter(F.col("id_avaliacao").isNotNull())
    .filter(F.col("nota_avaliacao").isNotNull())
    #remove datas futuras
    .filter(F.col("data_criacao_avaliacao") <= data_atual)
    .filter(F.col("data_resposta_avaliacao") <= data_atual)
)

df_silver = (
    df_silver
    .withColumn(
        "titulo_comentario_avaliacao",
        F.when(
            F.col("titulo_comentario_avaliacao").isNull() | (F.trim(F.col("titulo_comentario_avaliacao")) == ""),
            F.lit("Sem título")
        ).otherwise(F.trim(F.col("titulo_comentario_avaliacao")))
    )
    .withColumn(
        "comentario_avaliacao",
        F.when(
            F.col("comentario_avaliacao").isNull() | (F.trim(F.col("comentario_avaliacao")) == ""),
            F.lit("Sem comentário")
        ).otherwise(F.trim(F.col("comentario_avaliacao")))
    )
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("id_avaliacao").isNull()).count()
print(f"Total de registros nulos: {nulls}")
sem_nota = df_silver.filter(F.col("nota_avaliacao").isNull()).count()
print(f"Total de registros sem notas: {sem_nota}")


assert nulls == 0, "Existem registros nulos na tabela"
assert sem_nota == 0, "Existem registros sem nota na tabela"


TABLE_NAME = "ecommerce_lakehouse.silver.fat_avaliacoes_pedidos"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)


Total de registros: 95307
Total de registros nulos: 0
Total de registros sem notas: 0


Criação de tabela de dimensão de produto

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_products")

df_silver = df_bronze.select(
    F.col("product_id").cast("string").alias("id_produto"),
    F.col("product_category_name").cast("string").alias("categoria_produto"),
    F.col("product_name_lenght").cast("int").alias("tamanho_nome_produto"),
    F.col("product_description_lenght").cast("int").alias("tamanho_descricao_produto"),
    F.col("product_photos_qty").cast("int").alias("quantidade_fotos"),
    F.col("product_weight_g").cast("int").alias("peso_produto_gramas"),
    F.col("product_length_cm").cast("int").alias("comprimento_centimetros"),
    F.col("product_height_cm").cast("int").alias("altura_centimetros"),
    F.col("product_width_cm").cast("int").alias("largura_centimetros"),
    F.col("timestamp_ingestion").cast("timestamp").alias("carimbo_data_ingestao"),
    F.col("timestamp_ingestion").alias("carimbo_data_ingestao_bronze")
)

window_dedup = Window.partitionBy("id_produto").orderBy(F.col("carimbo_data_ingestao").desc())

df_silver = (
    df_silver
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("id_produto").isNull()).count()
print(f"Total de registros nulos: {nulls}")
duplicates = total - df_silver.select("id_produto").distinct().count()
print(f"Total de duplicidades: {duplicates}")

assert nulls == 0, "Existem registros nulos na tabela"
assert duplicates == 0, "Existem duplicidades na tabela"

TABLE_NAME = "ecommerce_lakehouse.silver.dim_produtos"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)


Total de registros: 32951
Total de registros nulos: 0


Criação de tabela de dimensão de vendedores

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_sellers")

df_silver = df_bronze.select(
    F.col("seller_id").cast("string").alias("id_vendedor"),
    F.col("seller_zip_code_prefix").cast("string").alias("prefixo_cep"),
    F.col("seller_city").cast("string").alias("cidade"),
    F.col("seller_state").cast("string").alias("estado"),
    F.col("timestamp_ingestion").cast("timestamp").alias("carimbo_data_ingestao_bronze")
)

df_silver = (
    df_silver
    .withColumn("prefixo_cep", F.lpad(F.col("prefixo_cep"),5, "0"))
    .withColumn("cidade", F.upper(F.trim(F.col("cidade"))))
    .withColumn("estado", F.upper(F.trim(F.col("estado"))))
    .filter(F.col("id_vendedor").isNotNull())
)

window_dedup = Window.partitionBy("id_vendedor").orderBy(F.col("carimbo_data_ingestao_bronze").desc())

df_silver = (
    df_silver
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("id_vendedor").isNull()).count()
print(f"Total de registros nulos: {nulls}")
duplicates = total - df_silver.select("id_vendedor").distinct().count()
print(f"Total de duplicados: {duplicates}")
assert nulls == 0, "Existem registros nulos na tabela"
assert duplicates == 0, "Existem duplicidades na tabela"

TABLE_NAME = "ecommerce_lakehouse.silver.dim_vendedores"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)


Total de registros: 3095
Total de registros nulos: 0
Total de duplicados: 0


Criação de tabela de dimensão de categorias dos produtos

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_product_category_name_translation")

df_silver = df_bronze.select(
    F.col("product_category_name").cast("string").alias("nome_produto_pt"),
    F.col("product_category_name_english").cast("string").alias("nome_produto_en"),
    F.col("timestamp_ingestion").cast("timestamp").alias("carimbo_data_ingestao")
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("nome_produto_pt").isNull()).count()
print(f"Total de registros nulos: {nulls}")

assert nulls == 0, "Existem registros nulos na tabela"

TABLE_NAME = "ecommerce_lakehouse.silver.dim_categoria_produtos_traducao"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)

Total de registros: 71
Total de registros nulos: 0


Criação de tabela de dimensão da cotação do dólar na Silver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.table("ecommerce_lakehouse.bronze.tb_cotacao_dolar")

df_bronze = df_bronze.select(
    F.to_date("dataHoraCotacao").alias("data_cotacao"),
    F.col("vlCotacaoCompra").cast("double").alias("valor_cotacao_compra"),
)

data_min, data_max = df_bronze.selectExpr(
    "min(data_cotacao)", "max(data_cotacao)"
).first()

df_calendario = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{data_min}'),
            to_date('{data_max}'),
            interval 1 day
        )
    ) AS data_cotacao
""")

df_silver = df_calendario.join(df_bronze, on="data_cotacao", how="left")

df_silver = (
    #indica se é dia util ou não
    df_silver
    .withColumn("fl_dia_util",
        F.when(F.dayofweek("data_cotacao").isin(1, 7), F.lit(0))
         .otherwise(F.lit(1))
    )
)

window_ffill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_silver = (
    df_silver
    .withColumn(
        "valor_cotacao_compra_ffill",
        F.last("valor_cotacao_compra", ignorenulls=True).over(window_ffill)
    )
    .drop("valor_cotacao_compra")
    .withColumnRenamed("valor_cotacao_compra_ffill", "valor_cotacao_compra")
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total = df_silver.count()
print(f"Total de registros: {total}")
nulls = df_silver.filter(F.col("valor_cotacao_compra").isNull()).count()
print(f"Total de registros nulos: {nulls}")

TABLE_NAME = "ecommerce_lakehouse.silver.dim_cotacao_dolar"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de registros: 773
Total de registros nulos: 0


Criação de tabela fato de produtos totais na silver

In [0]:
from pyspark.sql import functions as F

df_pedidos    = spark.table("ecommerce_lakehouse.silver.fat_pedidos")
df_pagamentos = spark.table("ecommerce_lakehouse.silver.fat_pagamentos_pedidos")
df_cotacao    = spark.table("ecommerce_lakehouse.silver.dim_cotacao_dolar")

df_pagamentos_agg = (
    df_pagamentos
    .filter(F.col("valor") > 0)
    .groupBy("id_pedido")
    .agg(
        F.round(F.sum("valor"), 2).alias("valor_total_pago_brl"),
        F.countDistinct("tipo_pagamento").alias("qt_formas_pagamento"),
        F.max("parcelas").alias("max_parcelas"),
        # Forma de pagamento predominante (maior valor)
        F.first("tipo_pagamento", ignorenulls=True).alias("tipo_pagamento_principal"),
    )
)

df_silver = df_pedidos.join(df_pagamentos_agg, on="id_pedido", how="inner")

df_cotacao_slim = df_cotacao.select(
    F.col("data_cotacao"),
    F.col("valor_cotacao_compra"),
)

df_silver = df_silver.join(
    df_cotacao_slim,
    df_silver["data_compra"].cast("date") == df_cotacao_slim["data_cotacao"],
    how="left"
)

df_silver = df_silver.select(
    F.col("id_pedido"),
    F.col("id_consumidor"),
    F.col("status_pedido").alias("status"),
    F.col("data_compra").alias("data_pedido"),

    F.col("valor_total_pago_brl"),

    F.round(
        F.col("valor_total_pago_brl") / F.col("valor_cotacao_compra"), 2
    ).alias("valor_total_pago_usd"),
)

df_silver = (
    df_silver
    .filter(F.col("id_pedido").isNotNull())
    .filter(F.col("valor_total_pago_brl") > 0)
    .filter(F.col("valor_total_pago_usd").isNotNull())
)

df_silver = (
    df_silver
    .withColumn("data_referencia",   F.to_date(F.current_timestamp()))
    .withColumn("carimbo_data_carregado_silver", F.current_timestamp())
)

total           = df_silver.count()
print(f"Total de registros: {total}")
nulos           = df_silver.filter(F.col("id_pedido").isNull()).count()
duplicados      = total - df_silver.select("id_pedido").distinct().count()
nulos_usd       = df_silver.filter(F.col("valor_total_pago_usd").isNull()).count()

assert nulos   == 0, "Existem nulos em id_pedido!"
assert duplicados == 0, "Existem duplicados!"
assert nulos_usd  == 0, "Existem nulos em valor_total_pago_usd!"

TABLE_NAME = "ecommerce_lakehouse.silver.fat_pedido_total"

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

Total de registros: 99436

🔎 Amostra — valores BRL e USD:
+--------------------------------+--------+--------------------+--------------------+
|id_pedido                       |status  |valor_total_pago_brl|valor_total_pago_usd|
+--------------------------------+--------+--------------------+--------------------+
|e481f51cbdc54678b7cc49136f2d6af7|Entregue|38.71               |12.24               |
|53cdb2fc8bc7dce0b6741e2150273451|Entregue|141.46              |37.77               |
|47770eb9100c2d0c44946d9cf07ec65d|Entregue|179.12              |47.75               |
|949d5b44dbf5de918fe9c16f97b45f8a|Entregue|72.2                |22.02               |
|ad21c59c0840e6cb83a9ceb5573f8159|Entregue|28.62               |8.72                |
+--------------------------------+--------+--------------------+--------------------+
only showing top 5 rows


Otimização fisíca Delta das tabelas fato

In [0]:
print("\nIniciando otimização física Delta...")

tabelas_fato = {
    "ecommerce_lakehouse.silver.fat_pedido_total":       ["id_pedido", "data_pedido"],
    "ecommerce_lakehouse.silver.fat_pedidos":            ["id_pedido", "data_compra"],
    "ecommerce_lakehouse.silver.fat_pagamentos_pedidos": ["id_pedido"],
    "ecommerce_lakehouse.silver.fat_avaliacoes_pedidos": ["id_pedido"],
}

for tabela, colunas_zorder in tabelas_fato.items():
    zorder_cols = ", ".join(colunas_zorder)
    print(f"\n   Otimizando: {tabela}")
    spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({zorder_cols})")
    print(f"   Concluído!")

print("\nFim da otimização física Delta!")


Iniciando otimização física Delta...

   Otimizando: ecommerce_lakehouse.silver.fat_pedido_total
   Concluído!

   Otimizando: ecommerce_lakehouse.silver.fat_pedidos
   Concluído!

   Otimizando: ecommerce_lakehouse.silver.fat_pagamentos_pedidos
   Concluído!

   Otimizando: ecommerce_lakehouse.silver.fat_avaliacoes_pedidos
   Concluído!

Fim da otimização física Delta!
